# Longrun ozone-loss predictability from stratospheric dynamics

目标：用机器学习研究 longrun 中严重北极臭氧损耗事件的可预报性，但把问题尽量写成一个平流层动力学问题，而不是“把变量丢给 ML”。

这版 notebook 的定义是：

- **预报对象**：3-4 月 `60-90N`, `30-70 hPa` partial O3 的 5 日平滑异常平均损耗，以及最低 25% 的严重臭氧事件。
- **预报提前量**：从 1 月 1 日到 2 月 28 日，每天作为一个 cutoff，计算距离 3 月 1 日还有多少天。
- **动力信息分组**：
  - `VORTEX_STATE`: NAM 10/50 hPa 表示极涡强度与垂直耦合。
  - `WAVE_FORCING`: 100/50 hPa EP flux 和 100 hPa EP-flux divergence 表示行星波活动及其对涡旋的强迫。
  - `DYN_COUPLED`: 极涡状态 + 波活动。
  - `DYN_PLUS_PASSIVE_TRACER`: 动力 + 早冬/晚冬臭氧异常。这里 O3 不作为独立模型讨论，只作为平流层环流历史的 passive-tracer proxy。
- **模型**：从可解释到非线性，使用 L2 线性模型、浅随机森林、浅梯度提升树。

可预报性不是单个高分数，而是一个 operational 判据：在 B2000WCN out-of-fold 检验中同时满足 AUC、Brier skill、top-quartile alarm、回归相关和回归 MSE skill 门槛，并且至少连续 7 个 cutoff 都满足，才称为稳定可预报窗口。


In [ ]:

from __future__ import annotations

import json
import math
import os
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import pearsonr
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.metrics import brier_score_loss, mean_squared_error, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "ML").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside code_cleaned.")

REPO_ROOT = find_repo_root()
OUT_DIR = REPO_ROOT / "ML" / "outputs" / "dynamical_predictability"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTH_LENGTHS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
MONTH_START_DOY0 = np.concatenate([[0], np.cumsum(MONTH_LENGTHS)[:-1]])

TARGET_START_DOY = 60
CUTOFF_DOYS = list(range(1, 60))
SHORT_WINDOW_DAYS = 28
LONG_WINDOW_DAYS = 90
EVENT_FRACTION = 0.25
RANDOM_STATE = 42

print("repo:", REPO_ROOT)
print("outputs:", OUT_DIR)


## Dataset paths

Inputs are read from the cleaned longrun products under `/mnt/soclim0/public_data/weiji`.  The notebook writes only inside `code_cleaned/ML`.


In [ ]:

@dataclass(frozen=True)
class DatasetConfig:
    name: str
    o3_file: Path
    nam_file: Path
    ep_file: Path

DATASETS: Mapping[str, DatasetConfig] = {
    "B2000WCN001002": DatasetConfig(
        name="B2000WCN001002",
        o3_file=Path("/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/partial_O3/B2000WCN_partial_O3_all_ranges.nc"),
        nam_file=Path("/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/NAM/B2000WCN001002_Vertical_NAM.nc"),
        ep_file=Path("/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/EPflux_daily/all_waves/EPFLUX_all_waves_210yr_time_plev_lat.nc"),
    ),
    "BWCN": DatasetConfig(
        name="BWCN",
        o3_file=Path("/mnt/soclim0/public_data/weiji/BWCN/partial_O3/BWCN_partial_O3_all_ranges.nc"),
        nam_file=Path("/mnt/soclim0/public_data/weiji/BWCN/NAM/BWCN_Vertical_NAM.nc"),
        ep_file=Path("/mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/all_waves/EPFLUX_all_waves_24yr_time_plev_lat.nc"),
    ),
}

def require_inputs(configs: Iterable[DatasetConfig]) -> None:
    missing = []
    for cfg in configs:
        for path in [cfg.o3_file, cfg.nam_file, cfg.ep_file]:
            if not path.exists():
                missing.append(path)
    if missing:
        raise FileNotFoundError("Missing inputs:\n" + "\n".join(str(p) for p in missing))

require_inputs(DATASETS.values())
print("Input files found.")


## Helper functions

The functions below keep the time handling explicit. The longrun products are no-leap 365-day years; where CAM `date` exists, it is used directly.


In [ ]:

def doy_from_month_day(month: np.ndarray, day: np.ndarray) -> np.ndarray:
    return MONTH_START_DOY0[month.astype(int) - 1] + day.astype(int)

def month_day_from_doy(doy: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    doy0 = doy.astype(int) - 1
    month = np.searchsorted(np.cumsum(MONTH_LENGTHS), doy0, side="right") + 1
    day = doy0 - MONTH_START_DOY0[month - 1] + 1
    return month.astype(int), day.astype(int)

def month_day_label(month: Sequence[int], day: Sequence[int]) -> List[str]:
    return [f"{int(m):02d}-{int(d):02d}" for m, d in zip(month, day)]

def frame_from_numeric_time(n_time: int, year_start: int = 1) -> pd.DataFrame:
    idx = np.arange(n_time, dtype=int)
    year = idx // 365 + year_start
    doy = idx % 365 + 1
    month, day = month_day_from_doy(doy)
    return pd.DataFrame({
        "year": year.astype(int),
        "doy": doy.astype(int),
        "month": month,
        "day": day,
        "month_day": month_day_label(month, day),
        "abs_day": ((year - 1) * 365 + doy).astype(int),
    })

def frame_from_cam_date(date_values: np.ndarray) -> pd.DataFrame:
    date = date_values.astype(int)
    year = date // 10000
    month = (date % 10000) // 100
    day = date % 100
    doy = doy_from_month_day(month, day)
    return pd.DataFrame({
        "year": year.astype(int),
        "doy": doy.astype(int),
        "month": month.astype(int),
        "day": day.astype(int),
        "month_day": month_day_label(month, day),
        "abs_day": ((year - 1) * 365 + doy).astype(int),
    })

def detect_data_var(ds: xr.Dataset, candidates: Sequence[str]) -> str:
    for name in candidates:
        if name in ds.data_vars:
            return name
    numeric = [name for name, da in ds.data_vars.items() if da.ndim >= 1 and np.issubdtype(da.dtype, np.number)]
    if not numeric:
        raise ValueError(f"No numeric data variable found: {list(ds.data_vars)}")
    return numeric[0]

def select_pressure(da: xr.DataArray, coord_name: str, target_hpa: float) -> xr.DataArray:
    values = np.asarray(da[coord_name].values, dtype=float)
    target = target_hpa * 100.0 if np.nanmax(values) > 2000.0 else target_hpa
    nearest = float(values[np.nanargmin(np.abs(values - target))])
    return da.sel({coord_name: nearest})

def linear_trend(values: np.ndarray, x: np.ndarray) -> float:
    mask = np.isfinite(values) & np.isfinite(x)
    if mask.sum() < 5:
        return np.nan
    xx = x[mask].astype(float)
    yy = values[mask].astype(float)
    xx = xx - xx.mean()
    return float(np.polyfit(xx, yy, 1)[0])

def window_mean(df: pd.DataFrame, value_col: str, start_abs: int, end_abs: int, min_count: int) -> float:
    sub = df[(df["abs_day"] >= start_abs) & (df["abs_day"] <= end_abs)]
    values = sub[value_col].astype(float)
    if values.notna().sum() < min_count:
        return np.nan
    return float(values.mean())

def window_trend(df: pd.DataFrame, value_col: str, start_abs: int, end_abs: int, min_count: int) -> float:
    sub = df[(df["abs_day"] >= start_abs) & (df["abs_day"] <= end_abs)]
    values = sub[value_col].astype(float)
    if values.notna().sum() < min_count:
        return np.nan
    return linear_trend(values.to_numpy(), sub["abs_day"].to_numpy())

def cutoff_label(cutoff_doy: int) -> str:
    month, day = month_day_from_doy(np.array([cutoff_doy]))
    return f"{int(month[0]):02d}-{int(day[0]):02d}"


## Load physically interpretable daily indices

- O3 is used to define the target, and later only as a passive-tracer predictor inside the combined dynamical model.
- NAM at 10/30/50 hPa is used as a compact vortex-strength and vertical-coupling description.
- EP flux and EP-flux divergence are used as wave-activity forcing proxies.


In [ ]:

def load_o3_daily(cfg: DatasetConfig) -> pd.DataFrame:
    print(f"[load] {cfg.name}: O3 target/passive tracer", flush=True)
    with xr.open_dataset(cfg.o3_file, decode_times=False) as ds:
        var = detect_data_var(ds, ["O3_partial_60_90N_30_70hPa"])
        values = ds[var].astype("float64").load().values
        frame = frame_from_cam_date(ds["date"].load().values) if "date" in ds else frame_from_numeric_time(values.shape[0])
    frame["O3_30_70_60_90N_DU"] = values
    frame["O3_rm5"] = pd.Series(values).rolling(5, center=True, min_periods=3).mean().to_numpy()
    frame["O3_clim_rm5"] = frame.groupby("month_day")["O3_rm5"].transform("mean")
    frame["O3_anom_rm5"] = frame["O3_rm5"] - frame["O3_clim_rm5"]
    return frame

def load_nam_daily(cfg: DatasetConfig, levels_hpa=(10, 30, 50)) -> pd.DataFrame:
    print(f"[load] {cfg.name}: NAM vortex-state levels {levels_hpa}", flush=True)
    with xr.open_dataset(cfg.nam_file, decode_times=False) as ds:
        var = detect_data_var(ds, ["NAM_Vertical", "NAM", "nam_vertical", "nam"])
        da = ds[var]
        lev_name = "lev" if "lev" in da.coords else list(da.dims)[-1]
        columns = {}
        for level in levels_hpa:
            columns[f"NAM{level}"] = select_pressure(da, lev_name, float(level)).astype("float64").load().values
    frame = frame_from_numeric_time(len(next(iter(columns.values()))))
    for name, values in columns.items():
        frame[name] = values
    frame["NAM10_minus_NAM50"] = frame["NAM10"] - frame["NAM50"]
    frame["NAM30_minus_NAM50"] = frame["NAM30"] - frame["NAM50"]
    return frame

def load_ep_daily(cfg: DatasetConfig, levels_hpa=(50, 100)) -> pd.DataFrame:
    print(f"[load] {cfg.name}: EP flux wave forcing levels {levels_hpa}", flush=True)
    columns = {}
    with xr.open_dataset(cfg.ep_file, decode_times=False) as ds:
        for var in ["ep2", "div2"]:
            if var not in ds:
                continue
            da = ds[var]
            for level in levels_hpa:
                selected = select_pressure(da, "plev", float(level))
                if "lat" in selected.dims:
                    sub = selected.sel(lat=slice(40, 80))
                    if sub.sizes.get("lat", 0) == 0:
                        sub = selected.sel(lat=slice(80, 40))
                    weights = np.cos(np.deg2rad(sub["lat"]))
                    selected = sub.weighted(weights).mean("lat")
                columns[f"{var.upper()}_{level}hPa_40_80N"] = selected.astype("float64").load().values
    frame = frame_from_numeric_time(len(next(iter(columns.values()))))
    for name, values in columns.items():
        frame[name] = values
    return frame

def build_target_table(o3_df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    rows = []
    for year, group in o3_df.groupby("year"):
        ma = group[group["month"].isin([3, 4])].copy()
        if ma["O3_rm5"].notna().sum() < 50:
            continue
        min_idx = ma["O3_rm5"].idxmin()
        rows.append({
            "dataset": dataset_name,
            "year": int(year),
            "MA_O3_anom": float(ma["O3_anom_rm5"].mean()),
            "MA_O3_loss": float(-ma["O3_anom_rm5"].mean()),
            "MA_O3_min_DU": float(ma["O3_rm5"].min()),
            "MA_O3_min_doy": int(ma.loc[min_idx, "doy"]),
        })
    target = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    n_event = max(1, int(math.floor(EVENT_FRACTION * len(target))))
    target["low_o3_rank"] = target["MA_O3_min_DU"].rank(method="first", ascending=True)
    target["Low25_min_label"] = (target["low_o3_rank"] <= n_event).astype(int)
    target["n_event"] = n_event
    return target

def load_bundle(cfg: DatasetConfig) -> Dict[str, pd.DataFrame]:
    o3 = load_o3_daily(cfg)
    nam = load_nam_daily(cfg)
    ep = load_ep_daily(cfg)
    target = build_target_table(o3, cfg.name)
    print(f"[target] {cfg.name}: n_years={len(target)}, n_events={int(target['Low25_min_label'].sum())}", flush=True)
    return {"o3": o3, "nam": nam, "ep": ep, "target": target}

bundles = {name: load_bundle(cfg) for name, cfg in DATASETS.items()}
bundles["B2000WCN001002"]["target"].head()


## Feature groups

The split is meant to ask physical questions:

1. Does the vortex state alone contain enough information?
2. Does wave forcing alone contain enough information?
3. Does the vortex-wave coupled state produce a longer stable lead time?
4. Does adding O3 as a passive tracer of prior transport/chemistry sharpen the alarm without changing the dynamic framing?


In [ ]:

PREDICTOR_SETS: Mapping[str, List[str]] = OrderedDict([
    ("VORTEX_STATE", [
        "NAM10_short_mean",
        "NAM50_short_mean",
        "NAM50_short_trend",
        "NAM10_minus_NAM50_short_mean",
    ]),
    ("WAVE_FORCING", [
        "EP2_100hPa_long_mean",
        "EP2_50hPa_long_mean",
        "DIV2_100hPa_long_mean",
        "EP2_100hPa_short_mean",
        "EP2_100hPa_long_trend",
    ]),
    ("DYN_COUPLED", [
        "NAM10_short_mean",
        "NAM50_short_mean",
        "NAM50_short_trend",
        "NAM10_minus_NAM50_short_mean",
        "EP2_100hPa_long_mean",
        "EP2_50hPa_long_mean",
        "DIV2_100hPa_long_mean",
        "EP2_100hPa_short_mean",
        "EP2_100hPa_long_trend",
    ]),
    ("DYN_PLUS_PASSIVE_TRACER", [
        "NAM10_short_mean",
        "NAM50_short_mean",
        "NAM50_short_trend",
        "NAM10_minus_NAM50_short_mean",
        "EP2_100hPa_long_mean",
        "EP2_50hPa_long_mean",
        "DIV2_100hPa_long_mean",
        "EP2_100hPa_short_mean",
        "EP2_100hPa_long_trend",
        "O3_short_mean",
        "O3_short_trend",
    ]),
])

def build_features_for_cutoff(cutoff_doy: int, bundle: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    target = bundle["target"]
    o3 = bundle["o3"]
    nam = bundle["nam"]
    ep = bundle["ep"]
    rows = []
    for _, tr in target.iterrows():
        year = int(tr["year"])
        cutoff_abs = (year - 1) * 365 + cutoff_doy
        short_start = cutoff_abs - SHORT_WINDOW_DAYS + 1
        long_start = cutoff_abs - LONG_WINDOW_DAYS + 1
        row = {
            "dataset": tr["dataset"],
            "year": year,
            "cutoff_doy": int(cutoff_doy),
            "cutoff_label": cutoff_label(cutoff_doy),
            "lead_days_to_Mar1": int(TARGET_START_DOY - cutoff_doy),
            "MA_O3_loss": float(tr["MA_O3_loss"]),
            "MA_O3_min_DU": float(tr["MA_O3_min_DU"]),
            "MA_O3_min_doy": int(tr["MA_O3_min_doy"]),
            "low_o3_rank": float(tr["low_o3_rank"]),
            "Low25_min_label": int(tr["Low25_min_label"]),
        }
        for col in ["NAM10", "NAM50", "NAM10_minus_NAM50"]:
            row[f"{col}_short_mean"] = window_mean(nam, col, short_start, cutoff_abs, 14)
            row[f"{col}_short_trend"] = window_trend(nam, col, short_start, cutoff_abs, 14)
        row["EP2_100hPa_long_mean"] = window_mean(ep, "EP2_100hPa_40_80N", long_start, cutoff_abs, 45)
        row["EP2_50hPa_long_mean"] = window_mean(ep, "EP2_50hPa_40_80N", long_start, cutoff_abs, 45)
        row["DIV2_100hPa_long_mean"] = window_mean(ep, "DIV2_100hPa_40_80N", long_start, cutoff_abs, 45)
        row["EP2_100hPa_short_mean"] = window_mean(ep, "EP2_100hPa_40_80N", short_start, cutoff_abs, 14)
        row["EP2_100hPa_long_trend"] = window_trend(ep, "EP2_100hPa_40_80N", long_start, cutoff_abs, 45)
        row["O3_short_mean"] = window_mean(o3, "O3_anom_rm5", short_start, cutoff_abs, 14)
        row["O3_short_trend"] = window_trend(o3, "O3_anom_rm5", short_start, cutoff_abs, 14)
        rows.append(row)
    features = pd.DataFrame(rows)
    all_predictors = sorted({x for xs in PREDICTOR_SETS.values() for x in xs})
    return features.dropna(subset=all_predictors + ["MA_O3_loss", "Low25_min_label"]).reset_index(drop=True)

example = build_features_for_cutoff(1, bundles["B2000WCN001002"])
example.head()


## Models and operational skill metrics

The algorithms are deliberately small-sample friendly:

- `linear_l2`: Ridge regression + L2 logistic regression. Baseline and sign-stable.
- `random_forest`: shallow bagged trees for thresholds/interactions.
- `gradient_boosting`: shallow boosted trees for nonlinear additive structure.


In [ ]:

MODEL_FAMILIES = ["linear_l2", "random_forest", "gradient_boosting"]

def make_regressor(model_family: str):
    if model_family == "linear_l2":
        return Pipeline([("scaler", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-4, 4, 41)))])
    if model_family == "random_forest":
        return RandomForestRegressor(n_estimators=160, max_depth=4, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=2)
    if model_family == "gradient_boosting":
        return GradientBoostingRegressor(n_estimators=120, learning_rate=0.035, max_depth=2, min_samples_leaf=8, random_state=RANDOM_STATE)
    raise ValueError(model_family)

def make_classifier(model_family: str, y: np.ndarray):
    if model_family == "linear_l2":
        folds = min(5, int(np.bincount(y, minlength=2).min()))
        return Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegressionCV(Cs=np.logspace(-3, 3, 13), cv=max(3, folds), penalty="l2", solver="lbfgs", max_iter=5000, scoring="neg_brier_score", random_state=RANDOM_STATE))])
    if model_family == "random_forest":
        return RandomForestClassifier(n_estimators=160, max_depth=4, min_samples_leaf=5, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=2)
    if model_family == "gradient_boosting":
        return GradientBoostingClassifier(n_estimators=120, learning_rate=0.035, max_depth=2, min_samples_leaf=8, random_state=RANDOM_STATE)
    raise ValueError(model_family)

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mse = float(mean_squared_error(y_true, y_pred))
    clim_mse = float(mean_squared_error(y_true, np.full_like(y_true, np.mean(y_true), dtype=float)))
    if len(y_true) >= 3 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr, pval = pearsonr(y_true, y_pred)
    else:
        corr, pval = np.nan, np.nan
    return {
        "reg_rmse": float(np.sqrt(mse)),
        "reg_mse_skill_vs_climatology": float(1 - mse / clim_mse) if clim_mse > 0 else np.nan,
        "reg_correlation": float(corr),
        "reg_correlation_pvalue": float(pval),
    }

def top_fraction_alarm(y_true: np.ndarray, score: np.ndarray, fraction: float = EVENT_FRACTION) -> Dict[str, float]:
    valid = np.isfinite(score)
    y = y_true[valid].astype(int)
    s = score[valid].astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return {"top25_hit_rate": np.nan, "top25_false_alarm_rate": np.nan, "top25_precision": np.nan}
    alarm = np.zeros(len(y), dtype=int)
    n_alarm = max(1, int(math.floor(fraction * len(y))))
    alarm[np.argsort(-s)[:n_alarm]] = 1
    tp = int(((alarm == 1) & (y == 1)).sum())
    fp = int(((alarm == 1) & (y == 0)).sum())
    tn = int(((alarm == 0) & (y == 0)).sum())
    fn = int(((alarm == 0) & (y == 1)).sum())
    return {
        "top25_hit_rate": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "top25_false_alarm_rate": float(fp / (fp + tn)) if (fp + tn) else np.nan,
        "top25_precision": float(tp / (tp + fp)) if (tp + fp) else np.nan,
        "top25_tp": tp, "top25_fp": fp, "top25_tn": tn, "top25_fn": fn,
    }

def classification_metrics(y_true: np.ndarray, prob: np.ndarray) -> Dict[str, float]:
    valid = np.isfinite(prob)
    y = y_true[valid].astype(int)
    p = prob[valid].astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        out = {"cls_roc_auc": np.nan, "cls_brier_score": np.nan, "cls_brier_skill_vs_climatology": np.nan}
    else:
        brier = float(brier_score_loss(y, p))
        q = float(np.mean(y))
        brier_clim = q * (1 - q)
        out = {"cls_roc_auc": float(roc_auc_score(y, p)), "cls_brier_score": brier, "cls_brier_skill_vs_climatology": float(1 - brier / brier_clim) if brier_clim > 0 else np.nan}
    out.update(top_fraction_alarm(y_true, prob))
    return out

def evaluate_cv(df: pd.DataFrame, features: Sequence[str], model_family: str) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    sub = df.dropna(subset=list(features) + ["MA_O3_loss", "Low25_min_label"]).copy()
    x = sub[list(features)].to_numpy(dtype=float)
    y_reg = sub["MA_O3_loss"].to_numpy(dtype=float)
    y_cls = sub["Low25_min_label"].to_numpy(dtype=int)
    reg_pred = cross_val_predict(make_regressor(model_family), x, y_reg, cv=KFold(n_splits=min(5, len(sub)), shuffle=True, random_state=RANDOM_STATE))
    prob = np.full(len(sub), np.nan)
    counts = np.bincount(y_cls, minlength=2)
    if counts.min() >= 3:
        cv_cls = StratifiedKFold(n_splits=min(5, int(counts.min())), shuffle=True, random_state=RANDOM_STATE)
        prob = cross_val_predict(make_classifier(model_family, y_cls), x, y_cls, cv=cv_cls, method="predict_proba")[:, 1]
    metrics = {**regression_metrics(y_reg, reg_pred), **classification_metrics(y_cls, prob), "n_samples": int(len(sub)), "n_events": int(y_cls.sum())}
    return metrics, reg_pred, prob

def fit_predict_external(train_df: pd.DataFrame, external_df: pd.DataFrame, features: Sequence[str], model_family: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    train = train_df.dropna(subset=list(features) + ["MA_O3_loss", "Low25_min_label"]).copy()
    ext = external_df.dropna(subset=list(features)).copy()
    x_train = train[list(features)].to_numpy(dtype=float)
    x_ext = ext[list(features)].to_numpy(dtype=float)
    y_reg = train["MA_O3_loss"].to_numpy(dtype=float)
    y_cls = train["Low25_min_label"].to_numpy(dtype=int)
    reg = make_regressor(model_family)
    reg.fit(x_train, y_reg)
    reg_pred = reg.predict(x_ext)
    prob = np.full(len(ext), np.nan)
    if np.bincount(y_cls, minlength=2).min() >= 3:
        clf = make_classifier(model_family, y_cls)
        clf.fit(x_train, y_cls)
        prob = clf.predict_proba(x_ext)[:, 1]
    return ext, reg_pred, prob


## Run lead-time scan

This cell is the main experiment. It saves only compact final outputs; full intermediate prediction matrices are intentionally not written.


In [ ]:

def add_probability_ranks(pred: pd.DataFrame) -> pd.DataFrame:
    out = pred.copy()
    out["prob_rank_desc"] = np.nan
    out["prob_top25_alarm"] = False
    group_cols = ["dataset", "cutoff_doy", "predictor_set", "model_family"]
    for _, idx in out.groupby(group_cols).groups.items():
        sub = out.loc[idx]
        valid_idx = sub[sub["pred_low25_probability"].notna()].index
        if len(valid_idx) == 0:
            continue
        ranks = sub.loc[valid_idx, "pred_low25_probability"].rank(method="first", ascending=False)
        n_alarm = max(1, int(math.floor(EVENT_FRACTION * len(valid_idx))))
        out.loc[valid_idx, "prob_rank_desc"] = ranks
        out.loc[valid_idx, "prob_top25_alarm"] = ranks <= n_alarm
    return out

def is_usable(row: pd.Series) -> bool:
    return bool(
        row.get("cls_roc_auc", np.nan) >= 0.75
        and row.get("cls_brier_skill_vs_climatology", np.nan) > 0
        and row.get("top25_hit_rate", np.nan) >= 0.60
        and row.get("top25_false_alarm_rate", np.nan) <= 0.35
        and row.get("reg_correlation", np.nan) >= 0.50
        and row.get("reg_mse_skill_vs_climatology", np.nan) > 0
    )

def sustained_earliest_lead(sub: pd.DataFrame, streak: int = 7) -> float:
    ordered = sub.sort_values("lead_days_to_Mar1", ascending=False).reset_index(drop=True)
    usable = ordered["usable_predictability"].to_numpy(dtype=bool)
    leads = ordered["lead_days_to_Mar1"].to_numpy(dtype=int)
    for i in range(len(ordered) - streak + 1):
        if usable[i : i + streak].all():
            return float(leads[i])
    return np.nan

def summarize_windows(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, pset), sub in summary.groupby(["model_family", "predictor_set"]):
        usable = sub[sub["usable_predictability"]]
        rows.append({
            "model_family": model,
            "predictor_set": pset,
            "earliest_single_usable_lead_days": float(usable["lead_days_to_Mar1"].max()) if not usable.empty else np.nan,
            "earliest_7day_stable_usable_lead_days": sustained_earliest_lead(sub, 7),
            "best_auc": float(sub["cls_roc_auc"].max()),
            "best_reg_correlation": float(sub["reg_correlation"].max()),
            "best_reg_mse_skill": float(sub["reg_mse_skill_vs_climatology"].max()),
        })
    return pd.DataFrame(rows).sort_values(["earliest_7day_stable_usable_lead_days", "earliest_single_usable_lead_days", "best_auc"], ascending=False, na_position="last")

train_bundle = bundles["B2000WCN001002"]
bwcn_bundle = bundles["BWCN"]
summary_rows = []
external_rows = []
bwcn0008_rows = []

for cutoff_doy in CUTOFF_DOYS:
    label = cutoff_label(cutoff_doy)
    print(f"[scan] {label}, lead={TARGET_START_DOY - cutoff_doy} d", flush=True)
    train_features = build_features_for_cutoff(cutoff_doy, train_bundle)
    bwcn_features = build_features_for_cutoff(cutoff_doy, bwcn_bundle)
    for predictor_set, features in PREDICTOR_SETS.items():
        for model_family in MODEL_FAMILIES:
            metrics, _, _ = evaluate_cv(train_features, features, model_family)
            summary_rows.append({
                "dataset": "B2000WCN001002",
                "cutoff_doy": cutoff_doy,
                "cutoff_label": label,
                "lead_days_to_Mar1": TARGET_START_DOY - cutoff_doy,
                "predictor_set": predictor_set,
                "model_family": model_family,
                "features": ",".join(features),
                **metrics,
            })
            ext, ext_reg, ext_prob = fit_predict_external(train_features, bwcn_features, features, model_family)
            keep = ext[["dataset", "year", "cutoff_doy", "cutoff_label", "lead_days_to_Mar1", "MA_O3_loss", "MA_O3_min_DU", "MA_O3_min_doy", "low_o3_rank", "Low25_min_label"]].copy()
            keep["predictor_set"] = predictor_set
            keep["model_family"] = model_family
            keep["pred_MA_O3_loss"] = ext_reg
            keep["pred_low25_probability"] = ext_prob
            external_rows.append(keep)

skill_summary = pd.DataFrame(summary_rows)
skill_summary["usable_predictability"] = skill_summary.apply(is_usable, axis=1)
window_summary = summarize_windows(skill_summary)
external_predictions = add_probability_ranks(pd.concat(external_rows, ignore_index=True))
bwcn0008_trace = external_predictions[external_predictions["year"].astype(int) == 8].copy()

skill_summary.to_csv(OUT_DIR / "leadtime_skill_summary.csv", index=False)
window_summary.to_csv(OUT_DIR / "predictability_windows.csv", index=False)
bwcn0008_trace.to_csv(OUT_DIR / "bwcn0008_trace.csv", index=False)

window_summary


## External BWCN check and BWCN0008

The BWCN check is not a perfect independent observational test; it is a shorter longrun-style sample.  It is useful as a stress test for whether the B2000WCN-trained dynamical relationships flag known severe BWCN events, especially BWCN0008.


In [ ]:

def best_external_summary(pred: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, pset), sub in pred.groupby(["model_family", "predictor_set"]):
        metric_rows = []
        for _, g in sub.groupby("cutoff_doy"):
            y = g["Low25_min_label"].to_numpy(dtype=int)
            prob = g["pred_low25_probability"].to_numpy(dtype=float)
            reg = regression_metrics(g["MA_O3_loss"].to_numpy(dtype=float), g["pred_MA_O3_loss"].to_numpy(dtype=float))
            cls = classification_metrics(y, prob)
            metric_rows.append({**reg, **cls})
        m = pd.DataFrame(metric_rows)
        rows.append({
            "model_family": model,
            "predictor_set": pset,
            "best_auc": float(m["cls_roc_auc"].max()),
            "best_reg_correlation": float(m["reg_correlation"].max()),
            "best_top25_hit_rate": float(m["top25_hit_rate"].max()),
            "best_top25_false_alarm_rate": float(m["top25_false_alarm_rate"].min()),
        })
    return pd.DataFrame(rows).sort_values(["best_auc", "best_reg_correlation"], ascending=False)

external_summary = best_external_summary(external_predictions)
external_summary.to_csv(OUT_DIR / "bwcn_external_skill_summary.csv", index=False)

bwcn0008_alarm = (
    bwcn0008_trace[bwcn0008_trace["prob_top25_alarm"]]
    .groupby(["model_family", "predictor_set"])["lead_days_to_Mar1"]
    .max()
    .reset_index(name="earliest_top25_alarm_lead_days")
    .sort_values("earliest_top25_alarm_lead_days", ascending=False)
)
bwcn0008_alarm.to_csv(OUT_DIR / "bwcn0008_alarm_summary.csv", index=False)

external_summary


## Plots


In [ ]:

def plot_leadtime(summary: pd.DataFrame, metric: str, ylabel: str, outfile: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=False)
    axes = axes.ravel()
    for ax, predictor_set in zip(axes, PREDICTOR_SETS):
        sub0 = summary[summary["predictor_set"] == predictor_set]
        for model, sub in sub0.groupby("model_family"):
            sub = sub.sort_values("lead_days_to_Mar1")
            ax.plot(sub["lead_days_to_Mar1"], sub[metric], marker="o", markersize=3, linewidth=1.2, label=model)
        ax.set_title(predictor_set)
        ax.grid(True, alpha=0.25)
        ax.set_xlabel("Lead days before 1 March")
        ax.set_ylabel(ylabel)
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(outfile, dpi=180)
    plt.close(fig)

def plot_bwcn0008(trace: pd.DataFrame, outfile: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
    axes = axes.ravel()
    for ax, predictor_set in zip(axes, PREDICTOR_SETS):
        sub0 = trace[trace["predictor_set"] == predictor_set]
        for model, sub in sub0.groupby("model_family"):
            sub = sub.sort_values("lead_days_to_Mar1")
            ax.plot(sub["lead_days_to_Mar1"], sub["pred_low25_probability"], marker="o", markersize=3, linewidth=1.2, label=model)
        ax.set_title(predictor_set)
        ax.grid(True, alpha=0.25)
        ax.set_xlabel("Lead days before 1 March")
        ax.set_ylabel("BWCN0008 low-O3 probability")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(outfile, dpi=180)
    plt.close(fig)

plot_leadtime(skill_summary, "cls_roc_auc", "Low-O3 event AUC", OUT_DIR / "fig_leadtime_auc.png")
plot_leadtime(skill_summary, "reg_correlation", "Ozone-loss regression correlation", OUT_DIR / "fig_leadtime_reg_correlation.png")
plot_bwcn0008(bwcn0008_trace, OUT_DIR / "fig_bwcn0008_probability.png")

for png in ["fig_leadtime_auc.png", "fig_leadtime_reg_correlation.png", "fig_bwcn0008_probability.png"]:
    print(OUT_DIR / png)


## Write compact findings report


In [ ]:

def fmt(value, digits=3) -> str:
    if value is None or not np.isfinite(value):
        return "nan"
    return f"{float(value):.{digits}f}"

def markdown_table(df: pd.DataFrame, columns: Optional[Sequence[str]] = None) -> str:
    frame = df.loc[:, list(columns)].copy() if columns is not None else df.copy()
    if frame.empty:
        return "_No rows._"
    def cell(value) -> str:
        if isinstance(value, (float, np.floating)):
            return fmt(float(value))
        return str(value)
    headers = list(frame.columns)
    rows = [[cell(v) for v in row] for row in frame.to_numpy()]
    widths = [max(len(str(h)), *(len(row[i]) for row in rows)) for i, h in enumerate(headers)]
    lines = ["| " + " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)) + " |",
             "| " + " | ".join("-" * widths[i] for i in range(len(headers))) + " |"]
    lines.extend("| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(headers))) + " |" for row in rows)
    return "\n".join(lines)

report = []
report.append("# Dynamical ozone-loss predictability scan")
report.append("")
report.append("Severe event target: lowest quartile of March-April minimum 5-day-smoothed 60-90N, 30-70 hPa partial ozone. O3 is not evaluated as a standalone predictor set; when used, it is treated as a passive tracer within `DYN_PLUS_PASSIVE_TRACER`.")
report.append("")
report.append("## Stable B2000WCN lead windows")
report.append("")
report.append(markdown_table(window_summary))
report.append("")
report.append("## BWCN external check")
report.append("")
report.append(markdown_table(external_summary, ["model_family", "predictor_set", "best_auc", "best_reg_correlation", "best_top25_hit_rate", "best_top25_false_alarm_rate"]))
report.append("")
report.append("## BWCN0008 top-quartile alarms")
report.append("")
report.append(markdown_table(bwcn0008_alarm))
report.append("")
report.append("## Interpretation")
report.append("")
report.append("- The vortex-only and wave-only partitions test whether the early signal sits primarily in the vortex state or in wave forcing.")
report.append("- `DYN_COUPLED` tests the physical pathway in which wave forcing modulates the vortex and therefore later ozone loss.")
report.append("- `DYN_PLUS_PASSIVE_TRACER` tests whether ozone as a passive tracer of prior transport/chemistry sharpens alarms; it should not be interpreted as a separate non-dynamical source of predictability.")
report.append("- True vortex morphology is not yet diagnosed here. NAM vertical coupling is only a compact proxy; a next step would compute vortex moment diagnostics from geopotential height or PV-like fields.")

(OUT_DIR / "findings.md").write_text("\n".join(report) + "\n")
print((OUT_DIR / "findings.md").read_text())
